# 🚀 TheLook Ecommerce: Pipeline de Ingesta Industrial (Mirror)

Este notebook es una réplica exacta del proceso de producción `make_dataset.py`. Está diseñado para procesar los **2.4M de registros** de TheLook asegurando integridad temporal y privacidad.

### 🧪 Lo que tus compañeros auditarán aquí:
1. **Zero Leakage**: El uso de `CUTOFF_DATE` para no entrenar con datos del futuro.
2. **Ingeniería de Eventos**: Transformación de clicks en variables de comportamiento.
3. **Zero Trust**: Destrucción de PII mediante Hashing + Salt.

## 1. Configuración de Entorno y Llaves 🔑
Configura `KAGGLE_USERNAME`, `KAGGLE_KEY` y `USER_SALT` en la pestaña **Secrets** de Colab.

In [ ]:
!pip install kagglehub pandas numpy

import os
import pandas as pd
import numpy as np
import hashlib
import kagglehub
from google.colab import userdata

# CARGA DE SECRETOS
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    USER_SALT = userdata.get('USER_SALT')
    if not USER_SALT: raise ValueError("USER_SALT faltante.")
    print("✅ Llaves configuradas.")
except:
    print("❌ ERROR: Configura los Secrets KAGGLE_USERNAME, KAGGLE_KEY y USER_SALT.")

## 2. Descarga de Datos Crudos

In [ ]:
print("🚀 Descargando dataset (mustafakeser4/looker-ecommerce-bigquery-dataset)...")
path = kagglehub.dataset_download("mustafakeser4/looker-ecommerce-bigquery-dataset")

RAW_DIR = path

orders = pd.read_csv(os.path.join(RAW_DIR, "orders.csv"))
order_items = pd.read_csv(os.path.join(RAW_DIR, "order_items.csv"))
events = pd.read_csv(os.path.join(RAW_DIR, "events.csv"))
users = pd.read_csv(os.path.join(RAW_DIR, "users.csv"))
print("✓ Tablas principales cargadas.")

## 3. Integridad Temporal y Filtrado (Anti-Leakage) 🛡️

In [ ]:
order_items['created_at'] = pd.to_datetime(order_items['created_at'], format='mixed', utc=True)
events['created_at'] = pd.to_datetime(events['created_at'], format='mixed', utc=True)

MAX_DATE = order_items['created_at'].max()
CUTOFF_DATE = MAX_DATE - pd.Timedelta(days=120)

print(f"📅 Fecha máxima: {MAX_DATE}")
print(f"📅 Corte para Features: {CUTOFF_DATE}")

# Filtrado Upstream
valid_status = ['Complete', 'Shipped', 'Processing']
order_items_clean = order_items[(order_items['status'].isin(valid_status)) & (order_items['created_at'] <= CUTOFF_DATE)]
events_clean = events[events['created_at'] <= CUTOFF_DATE]
returns_clean = order_items[(order_items['status'] == 'Returned') & (order_items['created_at'] <= CUTOFF_DATE)]

## 4. Ingeniería de Variables (RFM + Comportamiento Web)

In [ ]:
print("🧮 Calculando RFM y Agregaciones...")

# 4.1 RFM
user_features = order_items_clean.groupby('user_id').agg(
    total_orders=('order_id', 'nunique'),
    total_items=('id', 'count'),
    total_revenue=('sale_price', 'sum'),
    first_purchase=('created_at', 'min'),
    last_purchase=('created_at', 'max')
).reset_index()

user_features['recency_days'] = (CUTOFF_DATE - user_features['last_purchase']).dt.days
user_features['customer_tenure_days'] = (CUTOFF_DATE - user_features['first_purchase']).dt.days
user_features['avg_days_between'] = np.where(
    user_features['total_orders'] > 1,
    (user_features['last_purchase'] - user_features['first_purchase']).dt.days / (user_features['total_orders'] - 1),
    0
)

# 4.2 Devoluciones
returns_agg = returns_clean.groupby('user_id').size().reset_index(name='return_count')
user_features = user_features.merge(returns_agg, on='user_id', how='left').fillna(0)
user_features['return_rate'] = user_features['return_count'] / user_features['total_items']

# 4.3 Comportamiento Web (2.4M eventos)
print("🌐 Minería de eventos web...")
event_pivot = events_clean.groupby(['user_id', 'event_type']).size().unstack(fill_value=0).reset_index()
event_pivot.columns = [f"events_{c}" if c != 'user_id' else c for c in event_pivot.columns]
user_features = user_features.merge(event_pivot, on='user_id', how='left').fillna(0)

# 4.4 Demografía
user_demo = users[['id', 'age', 'gender', 'country', 'traffic_source']].rename(columns={'id': 'user_id'})
user_features = user_features.merge(user_demo, on='user_id', how='left')

## 5. Exportación y Descarga 📥

In [ ]:
from google.colab import files

# 5.1 Target Churn (Ventana Futura)
p_post = order_items[(order_items['status'].isin(valid_status)) & (order_items['created_at'] > CUTOFF_DATE)]['user_id'].unique()
user_features['is_churned'] = (~user_features['user_id'].isin(p_post)).astype(int)

# 5.2 Hashing SHA-256 + Salt
def salt_hash(id_val):
    hash_input = f"{id_val}{USER_SALT}".encode()
    return hashlib.sha256(hash_input).hexdigest()[:12]

print("🔒 Aplicando seudonimización...")
user_features['Internal_ID'] = user_features['user_id'].apply(salt_hash)
user_features = user_features.drop(columns=['user_id'])

# 5.3 Definir y Guardar el archivo
OUTPUT_FILE = "user_features_churn_colab.csv"
user_features.to_csv(OUTPUT_FILE, index=False)
print(f"✅ Proceso finalizado. Filas: {len(user_features)}")

# 5.4 Ejecutar Descarga
files.download(OUTPUT_FILE)